# 01. NB_NFL_INGESTION

## Ingesta de datos RAW desde nflverse hacia LH_RAW_NFL

Este notebook implementa el proceso de ingesta de datos de **[nflverse](https://github.com/nflverse)** hacia el Lakehouse **LH_RAW_NFL** de Microsoft Fabric.

Los archivos se almacenan en su formato original (Parquet), conservando sus nombres y sin aplicar transformaciones sobre su contenido.

### Características del proceso

- **Ingesta incremental:** descarga los archivos que todavía no existen en el almacenamiento y actualiza aquellos conjuntos de datos definidos como sujetos a actualización.
- **Carga histórica inicial:** permite descargar los datos históricos disponibles desde los años de inicio definidos para cada conjunto de datos.
- **Actualizaciones periódicas:** permite ejecutar nuevamente el proceso para incorporar información reciente.
- **Conservación de datos originales:** los archivos se almacenan sin aplicar limpieza, normalización ni transformaciones de negocio.

### Fuente de datos

La información se obtiene del ecosistema de datos abiertos de [nflverse](https://github.com/nflverse), que proporciona diferentes conjuntos de datos en formatos como CSV, Parquet y RDS.

Para este proyecto se seleccionó **Parquet** como formato de almacenamiento, considerando su naturaleza columnar y su compatibilidad con Apache Spark y Microsoft Fabric.

La selección del formato forma parte de las decisiones de diseño de la solución de ingeniería de datos, considerando aspectos como el tamaño de los archivos, el almacenamiento y la eficiencia de procesamiento.

### Actualización de los datos

Algunos conjuntos de datos reciben actualizaciones periódicas durante la temporada de la NFL. Por ejemplo, los datos de *Snap Counts* pueden actualizarse diariamente durante el desarrollo de la temporada.

Por esta razón, el proceso contempla diferentes políticas de actualización según el tipo de conjunto de datos.

**Referencia:** [nflverse-data](https://github.com/nflverse/nflverse-data)

**Estado de referencia del proyecto:** temporada NFL 2026, semana 8.


In [1]:
import os, time, requests
from datetime import datetime

StatementMeta(, 12a17af0-58f3-47b6-a20d-3af88cdcde66, 3, Finished, Available, Finished, False)

##### 1. Configuración

Definición de las fuentes de datos, parámetros de ejecución, directorio de almacenamiento y temporada de referencia.

In [2]:

URL = "https://github.com/nflverse/nflverse-data/releases/download"
BASE_PATH = "/lakehouse/default/Files/nflverse"

# fecha actual - now temp
TEMPORADA_ACTUAL = datetime.now().year
print("Año detectado:", TEMPORADA_ACTUAL)

DRY_RUN = False

FUENTES = {
    "play_by_play":   ("pbp",            "play_by_play_{y}.parquet",     1999),
    "rosters":        ("rosters",        "roster_{y}.parquet",           1999),
    "weekly_rosters": ("weekly_rosters", "roster_weekly_{y}.parquet",    2002),
    "stats_player":   ("stats_player",   "stats_player_week_{y}.parquet",1999),
    "stats_team":     ("stats_team",     "stats_team_week_{y}.parquet",  1999),
    "schedules":      ("schedules",      "games.parquet",                None),
    "players":        ("players",        "players.parquet",              None),
    "teams":          ("teams",          "teams_colors_logos.parquet",   None),
}


StatementMeta(, 12a17af0-58f3-47b6-a20d-3af88cdcde66, 4, Finished, Available, Finished, False)

Año detectado: 2026


##### 2. Construcción de enlaces

Generación dinámica de las URL de descarga a partir de las fuentes configuradas y sus respectivos patrones de nombres.

In [3]:

LINKS = []

for carpeta, (tag, patron, desde) in FUENTES.items():
    if desde is None:
        LINKS.append((carpeta, patron, f"{URL}/{tag}/{patron}"))
    else:
        for y in range(desde, TEMPORADA_ACTUAL + 1):
            nombre = patron.format(y=y)
            LINKS.append((carpeta, nombre, f"{URL}/{tag}/{nombre}"))

print(f"{len(LINKS)} links construidos. Ejemplos:")
for carpeta, nombre, url in LINKS[:2] + LINKS[-2:]:
    print(f" [{carpeta}] {url}")

if not os.path.exists("/lakehouse/default/"):
    raise RuntimeError("Falta poner LH_RAW_NFL como lakehouse por defecto.")


StatementMeta(, 12a17af0-58f3-47b6-a20d-3af88cdcde66, 5, Finished, Available, Finished, False)

140 links construidos. Ejemplos:
 [play_by_play] https://github.com/nflverse/nflverse-data/releases/download/pbp/play_by_play_1999.parquet
 [play_by_play] https://github.com/nflverse/nflverse-data/releases/download/pbp/play_by_play_2000.parquet
 [players] https://github.com/nflverse/nflverse-data/releases/download/players/players.parquet
 [teams] https://github.com/nflverse/nflverse-data/releases/download/teams/teams_colors_logos.parquet


##### 3. Lógica de validación

Definición de las condiciones que determinan si un archivo debe descargarse nuevamente o si puede conservarse la versión existente.

In [4]:

def debe_bajar(carpeta, nombre, ruta):
    if not os.path.exists(ruta):                 # si no existe el archivo se bajara
        return True
    
    es_archivo_unico = FUENTES[carpeta][2] is None
    if es_archivo_unico:                         # si es un archivo unico (schedules, players, teams), se actualizara "siempre"
        return True
    
    if str(TEMPORADA_ACTUAL) in nombre:          # si es la temporada actual, siempre se actualiza
        return True

    return False                                 # si es de una temporada pasada y ya existe, se salta


StatementMeta(, 12a17af0-58f3-47b6-a20d-3af88cdcde66, 6, Finished, Available, Finished, False)

##### 4. Descarga y validación de archivos

Ejecución de las solicitudes HTTP, descarga de archivos Parquet por bloques y comprobaciones de integridad antes de almacenarlos en su ubicación definitiva.

In [5]:

ok, saltados, no_existe, errores = 0, 0, [], []

for carpeta, nombre, url in LINKS:
    os.makedirs(f"{BASE_PATH}/{carpeta}", exist_ok=True)
    ruta = f"{BASE_PATH}/{carpeta}/{nombre}"
    temp_path = f"{ruta}.part"

    if not debe_bajar(carpeta, nombre, ruta):
        saltados += 1
        continue
    
    if DRY_RUN:
        print(f"(dry-run) bajaría {url}")
        continue
    
    try:
        bytes_descargados = 0

        # Descarga HTTP
        with requests.get(url, stream=True, timeout=300) as r:

            if r.status_code == 404:
                no_existe.append(url)
                continue
        
            r.raise_for_status()

            if r.status_code != 200:
                raise ValueError(f"Respuesta HTTP inesperada: {r.status_code}")

            # Tamaño esperado
            content_length = r.headers.get("content-length")
            content_encoding = r.headers.get("content-encoding", "identity")

            # Descarga en bloques de 1MB a un archivo temporal
            with open(temp_path, "wb") as f:
                for ch in r.iter_content(1024 * 1024):
                    if ch:
                        f.write(ch)
                        bytes_descargados += len(ch)

        # Comprobar: se recibieron datos?
        if bytes_descargados < 8:
            raise ValueError("Archivo vacío o demasiado pequeño")

        # Compara tamaño cuando Content-Length está disponible
        if content_length is not None and content_encoding.lower() == "identity":
            if bytes_descargados != int(content_length):
                raise ValueError("La descarga está incompleta")

        # Comprobar la firma .parquet
        with open(temp_path, "rb") as f:
            header = f.read(4)
            f.seek(-4, os.SEEK_END)
            footer = f.read(4)

        if header != b"PAR1" or footer != b"PAR1":
            raise ValueError("El archivo no tiene una firma Parquet válida")

        # Reemplazar el archivo definitivo solo si la descarga terminó bien
        os.replace(temp_path, ruta)

        ok += 1
        print(f"OK {carpeta}/{nombre} ({bytes_descargados / 1e6:.2f} MB)")
        time.sleep(0.2) # pausa que evitaria saturación

    except Exception as e:
        # Elimina solo el archivo temporal.
        if os.path.exists(temp_path):
            os.remove(temp_path)

        errores.append((url, str(e)))


StatementMeta(, 12a17af0-58f3-47b6-a20d-3af88cdcde66, 7, Finished, Available, Finished, False)

OK play_by_play/play_by_play_2026.parquet (8.07 MB)
OK rosters/roster_2026.parquet (0.56 MB)
OK weekly_rosters/roster_weekly_2026.parquet (0.84 MB)
OK stats_player/stats_player_week_2026.parquet (0.42 MB)
OK stats_team/stats_team_week_2026.parquet (0.09 MB)
OK schedules/games.parquet (0.52 MB)
OK players/players.parquet (3.40 MB)
OK teams/teams_colors_logos.parquet (0.01 MB)


##### 5. Resumen de ejecución

Presentación de los resultados del proceso de ingesta: archivos descargados, archivos omitidos, recursos no encontrados y errores registrados.

In [6]:

print(f"\n--- RESUMEN ---")
print(f"Descargados: {ok} | Saltados: {saltados} | 404: {len(no_existe)} | Errores: {len(errores)}")
for u in no_existe: print(" 404:", u)
for u, e in errores: print(" ERROR:", u, e)

StatementMeta(, 12a17af0-58f3-47b6-a20d-3af88cdcde66, 8, Finished, Available, Finished, False)


--- RESUMEN ---
Descargados: 8 | Saltados: 132 | 404: 0 | Errores: 0
